# Deep Q-learning on the lunar lander

A port of the deep Q-learning half of `lunardeepQSarsaSemiGradient.py` -- the
code behind the result plot on the deck's slide 16.

This is Unit 10's Q-learning with the table thrown away. The state is an
$8$-dimensional continuous vector, so there is no table to keep: the Q-factors
are a neural network, and the Bellman error is the regression target of the
previous two notebooks. Everything else about the algorithm is unchanged.

What *is* new are the two devices that make it work in practice, both on the
deck's "special features" slide:

- an **experience replay** buffer, so that consecutive highly-correlated
  transitions are not used in the order they arrived;
- a separate, slowly updated **target network**, so that the target of the
  regression does not move as fast as the thing being fitted.

The notebook runs with those two devices and then, at the end, without them,
which is the quickest way to see what they are for.

::: Requirements :::
`torch` and `gymnasium`. The lunar lander itself needs Box2D
(`pip install "gymnasium[box2d]"`); without it the helper falls back to
`CartPole-v1`, which ships with the base package and exercises exactly the
same code. The agent code lives in `deep_rl.py` next to this notebook.

**Convention warning.** Gymnasium hands out *rewards to be maximised* while
Units 1-11 minimise a *cost*. The greedy action below is an `argmax` where
every formula in the notes is an `argmin`; the two differ by a sign.

In [ ]:
import sys, time
import numpy as np
import matplotlib.pyplot as plt

# deep_rl.py sits next to this notebook -- download it alongside
sys.path.insert(0, ".")
import deep_rl
from deep_rl import Config, seed_all, make_env, DQNAgent, SARSAAgent, \
                    train_dqn, train_sarsa, evaluate_agent, smooth

NUM_EPISODES = 800      # the deck's figures use 800; 150-200 already shows the shape
cfg = Config(num_episodes=NUM_EPISODES)
print("device:", deep_rl.DEVICE)

## The environment

Four discrete actions -- do nothing, fire left engine, fire main engine, fire
right engine -- and an eight-dimensional state: position $(x,y)$, angle
$\theta$, the three corresponding velocities, and two booleans for whether
each leg is touching the ground.

The reward is shaped (closer to the pad, slower, more upright, legs down all
score better), with $\pm100$ for landing or crashing and a small penalty per
frame of engine use. An episode scoring $200$ or more counts as solved.

In [ ]:
seed_all(cfg.seed)
env = make_env(cfg.env_id, seed=cfg.seed)
eval_env = make_env(cfg.env_id, seed=cfg.seed + 100, quiet=True)
obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n
print(f"environment  {env.spec.id}")
print(f"observation  {obs_dim} dimensions")
print(f"actions      {n_actions}")

## The agent

The network is three linear layers of $128$ units with ReLU between them --
the deck's "three-layer NN". The hyperparameters are those of the original
script:

| | |
|:--|:--|
| discount | $0.99$ |
| step size (Adam) | $10^{-3}$ |
| minibatch | $64$ |
| replay capacity | $50\,000$ transitions |
| training starts after | $1\,000$ steps |
| target network copied every | $1\,000$ steps |
| $\epsilon$ | $1.0 \to 0.02$, decayed by $0.995$ per episode |

The $\epsilon$ schedule is the Unit 11 exploration story: start by acting at
random, and let the greedy action take over as the estimates become worth
trusting. With a decay of $0.995$ per episode, $\epsilon$ reaches its floor
of $0.02$ after about $780$ episodes -- so exploration is still meaningful
across almost the whole run.

In [ ]:
agent = DQNAgent(obs_dim, n_actions, cfg)
print(agent.q)
print("parameters:", sum(p.numel() for p in agent.q.parameters()))
eps_floor = next(e for e in range(2000)
                 if cfg.dqn_eps_start * cfg.dqn_eps_decay**e <= cfg.dqn_eps_end)
print(f"epsilon reaches {cfg.dqn_eps_end} after {eps_floor} episodes")

## Training

One gradient step per environment step. Expect a few minutes.

In [ ]:
t0 = time.perf_counter()
dqn_returns, dqn_evals = train_dqn(agent, env, cfg, eval_env=eval_env, verbose=True)
print(f"\ntrained {len(dqn_returns)} episodes in {time.perf_counter()-t0:.0f} s")

## The result

This is the figure on slide 16: the raw per-episode return against episode
number, with a moving average on top so the trend is visible through the
noise.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(dqn_returns, color="C0", alpha=0.35, lw=1, label="DQN returns (train)")
sm = smooth(dqn_returns, 20)
ax.plot(np.arange(len(sm)) + 19, sm, color="C3", lw=2, label="moving average (20)")
ax.axhline(200, color="k", ls=":", lw=1)
ax.text(2, 210, "solved (200)", fontsize=9)
ax.set_title("DQN Training Returns")
ax.set_xlabel("Episode"); ax.set_ylabel("Episode return")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

The shape to look for, and the reason the raw curve is kept rather than
only the average:

**The scatter never goes away.** Even late in training, individual episodes
still land anywhere from $-200$ to $+300$. Part of that is the environment --
the lander starts with a random push, so some episodes are simply harder --
and part is the $2\%$ residual exploration, where a random main-engine burn
at the wrong moment costs an otherwise good landing. A learning curve that
became a clean line would be the suspicious one.

**Progress is not monotone.** Stretches of improvement are interrupted by
sharp drops. This is the price of bootstrapping on a moving approximation:
the target network is refreshed every $1\,000$ steps, and each refresh shifts
the target that the network is chasing. Unit 10's convergence theorem does not
apply here at all -- it was proved for the tabular case, and with function
approximation deep Q-learning has no comparable guarantee.

In [ ]:
mean, std = evaluate_agent(agent, eval_env, episodes=20)
print(f"greedy policy over 20 evaluation episodes: {mean:.1f} +/- {std:.1f}")
print(f"best training episode: {max(dqn_returns):.1f}")
print(f"mean of last 100 training episodes: {np.mean(dqn_returns[-100:]):.1f}")
print(f"fraction of the last 100 episodes scoring >= 200: "
      f"{np.mean(np.array(dqn_returns[-100:]) >= 200):.0%}")

## What the two special features buy

The claim on the deck is that experience replay breaks correlations and the
target network prevents oscillation. Both are easy to test: rerun with the
replay buffer shrunk to the minibatch size (so each sample is used almost
immediately, in order) and with the target network copied every step (so the
target chases the estimate).

This is a short run -- enough to see the difference in the early curve, not a
full training.

In [ ]:
ABLATION_EPISODES = min(200, NUM_EPISODES)

def run_variant(label, **overrides):
    c = Config(num_episodes=ABLATION_EPISODES, **overrides)
    seed_all(cfg.seed)
    e = make_env(cfg.env_id, seed=cfg.seed, quiet=True)
    a = DQNAgent(e.observation_space.shape[0], e.action_space.n, c)
    r, _ = train_dqn(a, e, c, num_episodes=ABLATION_EPISODES, verbose=False)
    e.close()
    print(f"{label:<34} mean of last 50: {np.mean(r[-50:]):8.1f}")
    return r

base_r    = run_variant("full algorithm")
noreplay  = run_variant("replay buffer ~ one minibatch",
                        dqn_replay_size=64, dqn_start_train=64)
notarget  = run_variant("target network copied every step",
                        dqn_target_update=1)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for r, lab in ((base_r, "full algorithm"),
               (noreplay, "replay buffer ~ one minibatch"),
               (notarget, "target network copied every step")):
    sm = smooth(r, 20)
    ax.plot(np.arange(len(sm)) + 19, sm, lw=2, label=lab)
ax.set_xlabel("Episode"); ax.set_ylabel("Episode return (moving average, 20)")
ax.set_title(f"Deep Q-learning, first {ABLATION_EPISODES} episodes")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

Read the numbers printed above rather than the exact curves -- a single
seed over a couple of hundred episodes is noisy, and the gaps are not always
dramatic this early. The direction is the point: both devices exist to stop
the regression target from moving in step with the estimate being fitted, and
removing either one makes the early learning less reliable.

Note what is *not* being claimed. These are not two tricks that happen to
help; they are what replaces the independence and stationarity assumptions
that the tabular convergence proof of Unit 10 relied on and that a stream of
consecutive transitions from a changing policy flatly violates.

## Optional: watch the learned policy

The original script also saves an animation of a single greedy episode
(`dqn_policy.gif`, linked from slide 16). This needs `imageio` and a render
mode the environment supports; it is skipped silently if either is
unavailable.

In [ ]:
try:
    import imageio
    render_env = make_env(cfg.env_id, render_mode="rgb_array", quiet=True)
    obs, _ = render_env.reset(seed=cfg.seed)
    frames, done, steps = [], False, 0
    while not done and steps < 600:
        obs, r, term, trunc, _ = render_env.step(agent.select_action(obs, eval_mode=True))
        done = term or trunc
        frame = render_env.render()
        if frame is not None:
            frames.append(frame)
        steps += 1
    render_env.close()
    if frames:
        imageio.mimsave("dqn_policy.gif", frames, fps=30)
        print(f"saved dqn_policy.gif ({len(frames)} frames)")
    else:
        print("no frames captured -- rendering not available here")
except Exception as exc:
    print("animation skipped:", type(exc).__name__, exc)